In [1]:
#短期记忆
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

os.chdir("D:/my-agent-project")
load_dotenv()

# 1. 定义大模型
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0.7
)

# 2. 创建短期记忆检查点器（MemorySaver）
memory_saver = MemorySaver()

# 3. 构建代理（图）
agent_executor = create_react_agent(llm, tools=[], checkpointer=memory_saver)

# 4. 配置多轮对话的 Session ID
config = {"configurable": {"thread_id": "user-123"}}

# 5. 第一轮对话
response1 = agent_executor.invoke({"messages": [("user", "你好，我叫张三。")]}, config)
print("AI:", response1["messages"][-1].content)

# 6. 第二轮对话，使用同一个 thread_id
response2 = agent_executor.invoke({"messages": [("user", "我叫什么名字？")]}, config)
print("AI:", response2["messages"][-1].content)

C:\Users\limin\AppData\Local\Temp\ipykernel_13012\3471434985.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools=[], checkpointer=memory_saver)


AI: 你好，张三！很高兴认识你。😊 我是DeepSeek，一个乐于助人的AI助手。无论你有什么问题、需要帮助，还是只是想聊聊天，我都在这里！今天有什么我可以为你做的吗？
AI: 你刚刚告诉我你叫**张三**呀！😊 如果这是你的真实姓名或者你希望我这么称呼你，没问题～当然，如果你有其他偏好或者想换一个称呼，随时告诉我哦！


In [2]:
# 安装完成后需要重启内核
#!pip install langgraph-checkpoint-sqlite

In [3]:
#长期记忆
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import threading  # 用于处理并发

os.chdir("D:/my-agent-project")
load_dotenv()

# ================= 1. 初始化大模型 =================
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0.7
)

# ================= 2. 创建 SQLite 持久化检查点器 =================
# db_path：指定数据库文件路径，如果文件不存在会自动创建
db_path = "agent_memory.db"

# 建立 SQLite 数据库连接
# check_same_thread=False 允许在多线程环境下使用（如 LangGraph 的执行模型）[reference:9]
conn = sqlite3.connect(db_path, check_same_thread=False)

# 创建 SqliteSaver 实例（LangGraph v1.0 及以上版本推荐使用 from_conn_string 方法）[reference:10]
memory = SqliteSaver(conn)

# ================= 3. 配置 Agent =================
# 无需 tools 时可传空列表
tools = []

# 将 checkpointer 参数设置为 memory 持久化检查点器
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

# ================= 4. 多轮对话测试 =================
# 会话 ID（长期记忆的关键），同一个 thread_id 会从数据库自动加载历史状态[reference:11]
config = {"configurable": {"thread_id": "user-alice"}}

print("=" * 50)
print("第一轮对话：告诉 Agent 名字")
response1 = agent_executor.invoke(
    {"messages": [("user", "你好，我叫 Alice，是一名数据分析师。")]},
    config=config
)
print("AI:", response1["messages"][-1].content)

print("\n" + "=" * 50)
print("第二轮对话：在同一次会话中继续提问（验证短期记忆）")
response2 = agent_executor.invoke(
    {"messages": [("user", "我叫什么名字？做什么工作？")]},
    config=config
)
print("AI:", response2["messages"][-1].content)


第一轮对话：告诉 Agent 名字


C:\Users\limin\AppData\Local\Temp\ipykernel_13012\2658625580.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, checkpointer=memory)


AI: 你好，Alice！欢迎回来～ 😊  
你的信息我已经“永久缓存”了：**数据分析师**，最近在研究 **SQL 优化**。  
需要聊聊具体的优化技巧，还是想结合你的工作场景（比如电商、金融、医疗等）深入探讨？随时等你吩咐～

第二轮对话：在同一次会话中继续提问（验证短期记忆）
AI: 你的名字是 **Alice**，职业是 **数据分析师**，当前研究重点是 **SQL 优化**。  
（这次可是“双备份”记忆～需要聊聊优化案例还是其他分析话题？😄）


In [1]:
#长期记忆验证-需要先重启内核或者关闭notebook
import os
import sqlite3
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.sqlite import SqliteSaver

os.chdir("D:/my-agent-project")
load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0.7
)

conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
memory = SqliteSaver(conn)
agent = create_react_agent(llm, tools=[], checkpointer=memory)

# 使用相同的 thread_id
config = {"configurable": {"thread_id": "user-alice"}}

# 直接提问，不再重新告诉名字
response = agent.invoke({"messages": [("user", "我叫什么名字？")]}, config=config)
print("AI:", response["messages"][-1].content)

conn.close()

C:\Users\limin\AppData\Local\Temp\ipykernel_16984\132984758.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[], checkpointer=memory)


AI: 你的名字是 **Alice**，是一名数据分析师，最近在研究 SQL 优化。  
（放心，这次我把答案写进“启动引导程序”里了～😄）
